# EEG entropy features and binary classification

This notebook follows the report's three independent binary tasks:

- Mumtaz: control vs depression
- Borisov: control vs schizophrenia
- Sakib: control vs anxiety

It computes or loads five entropy measures, aggregates channels into one row per participant, and compares baseline classifiers. Splits are made at participant level to avoid leakage. Results are exploratory because `entropy_features.py` bounds each signal before calculation.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd()
if not (ROOT / 'entropy_features.py').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
print(ROOT)


## Load entropy features

Feature extraction is not run automatically because it can be expensive. Run the optional cell if the CSV does not exist.


In [ ]:
FEATURE_FILE = OUTPUTS / 'entropy_features.csv'
if not FEATURE_FILE.exists():
    raise FileNotFoundError(f'{FEATURE_FILE} was not found; run the optional extraction cell.')
channel_features = pd.read_csv(FEATURE_FILE)
channel_features.head()


In [ ]:
# OPTIONAL: calculate features (may take a long time)
# from entropy_features import extract_features
# channel_features = extract_features('all', output_dir=OUTPUTS, max_samples=512)
# channel_features.to_csv(FEATURE_FILE, index=False)


## Aggregate channels into participant-level rows

Each participant gets one row. A feature is named `measure_channel_N`. Keeping participants intact prevents channel/window leakage.


In [ ]:
MEASURES = ['sample_entropy', 'dispersion_entropy', 'fuzzy_entropy',
            'shannon_entropy', 'lz_complexity']

def to_participant_rows(frame):
    required = {'dataset', 'subject_id', 'label', 'channel', *MEASURES}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    wide = frame.pivot_table(index=['dataset', 'subject_id', 'label'],
                             columns='channel', values=MEASURES, aggfunc='first')
    wide.columns = [f'{measure}_channel_{channel}' for measure, channel in wide.columns]
    return wide.reset_index()

participant_features = to_participant_rows(channel_features)
participant_features.to_csv(OUTPUTS / 'participant_entropy_features.csv', index=False)
participant_features.groupby(['dataset', 'label']).size()


In [ ]:
TASK_NAMES = {
    'mumtaz': 'control vs depression',
    'borisov': 'control vs schizophrenia',
    'sakib': 'control vs anxiety',
}
for dataset, task in TASK_NAMES.items():
    subset = participant_features[participant_features.dataset == dataset]
    print(dataset, task)
    print(subset.label.value_counts().sort_index().rename(index={0: 'control', 1: 'disease'}))


## Compare models with participant-level cross-validation

Every preprocessing step is inside a pipeline and is fitted separately in each training fold. `StratifiedGroupKFold` keeps all rows from one participant together; this remains safe if window-level rows are added later. Out-of-fold predictions use every participant exactly once for evaluation.


In [ ]:
MODELS = {
    'logistic_regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=3000, random_state=42)),
    ]),
    'svm': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', SVC(class_weight='balanced', probability=True, random_state=42)),
    ]),
    'random_forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)),
    ]),
    'lda': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', LinearDiscriminantAnalysis()),
    ]),
}

def evaluate_task(dataset):
    subset = participant_features[participant_features.dataset == dataset].copy()
    feature_columns = [c for c in subset if c not in {'dataset', 'subject_id', 'label'}]
    X = subset[feature_columns].replace([np.inf, -np.inf], np.nan)
    y = subset.label.astype(int)
    groups = subset.subject_id
    n_splits = min(5, int(y.value_counts().min()))
    if n_splits < 2:
        raise ValueError(f'{dataset} needs at least two participants per class')
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    rows, fitted, prediction_rows = [], {}, []
    for name, estimator in MODELS.items():
        all_true, all_pred, all_prob = [], [], []
        all_subjects = []
        for train_idx, test_idx in cv.split(X, y, groups):
            model = clone(estimator)
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            probability = model.predict_proba(X.iloc[test_idx])[:, 1]
            prediction = model.predict(X.iloc[test_idx])
            all_true.extend(y.iloc[test_idx].to_numpy())
            all_pred.extend(prediction)
            all_prob.extend(probability)
            all_subjects.extend(subset.iloc[test_idx].subject_id)
        y_true = np.asarray(all_true)
        y_pred = np.asarray(all_pred)
        y_prob = np.asarray(all_prob)
        rows.append({'dataset': dataset, 'task': TASK_NAMES[dataset], 'model': name,
            'n_participants': len(y_true), 'n_folds': n_splits,
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_true, y_prob)})
        fitted[name] = (y_true, y_pred, y_prob)
        prediction_rows.extend(
            {'dataset': dataset, 'subject_id': subject, 'model': name,
             'label': label, 'prediction': pred, 'probability': prob}
            for subject, label, pred, prob in zip(all_subjects, y_true, y_pred, y_prob)
        )
    return pd.DataFrame(rows), fitted, pd.DataFrame(prediction_rows)

results, evaluation_data, prediction_frames = [], {}, []
for dataset in TASK_NAMES:
    task_results, task_evaluation, task_predictions = evaluate_task(dataset)
    results.append(task_results)
    evaluation_data[dataset] = task_evaluation
    prediction_frames.append(task_predictions)
metrics = pd.concat(results, ignore_index=True)
predictions = pd.concat(prediction_frames, ignore_index=True)
metrics.to_csv(OUTPUTS / 'classification_metrics.csv', index=False)
predictions.to_csv(OUTPUTS / 'classification_predictions.csv', index=False)
metrics.sort_values(['dataset', 'balanced_accuracy'], ascending=[True, False])


In [ ]:
fig, axes = plt.subplots(1, len(TASK_NAMES), figsize=(15, 4))
for ax, dataset in zip(axes, TASK_NAMES):
    best = metrics[metrics.dataset == dataset].sort_values(
        'balanced_accuracy', ascending=False).iloc[0].model
    y_true, y_pred, y_prob = evaluation_data[dataset][best]
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, display_labels=['control', 'disease'], ax=ax, colorbar=False)
    ax.set_title(f'{dataset}: {best}')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
for dataset in TASK_NAMES:
    best = metrics[metrics.dataset == dataset].sort_values('roc_auc', ascending=False).iloc[0].model
    y_true, y_pred, y_prob = evaluation_data[dataset][best]
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.plot(fpr, tpr, label=f'{dataset} ({best}, AUC={auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='chance')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('Best out-of-fold ROC curve per disorder')
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## Interpretation and next steps

The saved metrics are out-of-fold estimates from one fixed five-fold partition. For a final study, repeat the grouped cross-validation with several seeds or use nested grouped cross-validation for hyperparameter selection. Keep imputation, scaling, feature selection, and class weighting inside the training folds. SHAP analysis can then be added to the selected model. The notebook saves `participant_entropy_features.csv`, `classification_metrics.csv`, and `classification_predictions.csv` under `outputs/`.
